In [1]:
!pip install pgmpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 23.1 MB/s eta 0:00:00


In [2]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

In [3]:
model = DiscreteBayesianNetwork([
    ('Income', 'LoanDefault'),
    ('CreditScore', 'LoanDefault'),
    ('Employment', 'LoanDefault'),
    ('LoanAmount', 'LoanDefault')
])

In [4]:
# 0 = Good/Low/Stable, 1 = Bad/High/Risky

# Income (0 = High, 1 = Low)
cpd_income = TabularCPD('Income', 2, [[0.6], [0.4]])

# Credit Score (0 = Good, 1 = Poor)
cpd_credit = TabularCPD('CreditScore', 2, [[0.7], [0.3]])

# Employment (0 = Stable, 1 = Unstable)
cpd_emp = TabularCPD('Employment', 2, [[0.65], [0.35]])

# Loan Amount (0 = Low, 1 = High)
cpd_loan = TabularCPD('LoanAmount', 2, [[0.5], [0.5]])

# Loan Default depends on all
cpd_default = TabularCPD(
    variable='LoanDefault',
    variable_card=2,
    values=[
        # No Default
        [0.95, 0.8, 0.75, 0.6, 0.7, 0.5, 0.4, 0.2,
         0.85, 0.7, 0.65, 0.5, 0.6, 0.4, 0.3, 0.1],

        # Default
        [0.05, 0.2, 0.25, 0.4, 0.3, 0.5, 0.6, 0.8,
         0.15, 0.3, 0.35, 0.5, 0.4, 0.6, 0.7, 0.9]
    ],
    evidence=['Income', 'CreditScore', 'Employment', 'LoanAmount'],
    evidence_card=[2, 2, 2, 2]
)

In [5]:
model.add_cpds(cpd_income, cpd_credit, cpd_emp, cpd_loan, cpd_default)

print("Model Valid:", model.check_model())

Model Valid: True


In [6]:
inference = VariableElimination(model)

In [7]:
def get_input(question):
    val = input(question + " (yes/no): ").strip().lower()
    return 1 if val == "yes" else 0

income = get_input("Is your income low?")
credit = get_input("Is your credit score poor?")
employment = get_input("Is your employment unstable?")
loan = get_input("Is your loan amount high?")

Is your income low? (yes/no): yes
Is your credit score poor? (yes/no): yes
Is your employment unstable? (yes/no): yes
Is your loan amount high? (yes/no): yes


In [8]:
evidence = {
    'Income': income,
    'CreditScore': credit,
    'Employment': employment,
    'LoanAmount': loan
}

result = inference.query(variables=['LoanDefault'], evidence=evidence)

print("\nLoan Default Risk:")
print(result)


Loan Default Risk:
+----------------+--------------------+
| LoanDefault    |   phi(LoanDefault) |
+================+====================+
| LoanDefault(0) |             0.1000 |
+----------------+--------------------+
| LoanDefault(1) |             0.9000 |
+----------------+--------------------+
